<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/6-4_nl2sql-ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>6.4-Setting UP a NL2SQL project with Ollama. </h2>
    <h3></h3>
    <p>by <b>Pere Martra</b></p>
</div>

Ollama is one of the simplest and easiest to configure model servers that you can use in your development environment.

In this notebook you are going to use models from Ollama and create a custom one able to generate SQL.

**This notebook needs a running Ollama server.** The setup cell below installs Ollama automatically and starts the server for you, so it works on Google Colab as well as on a local machine or an HPC node — no manual installation required. (On your own computer you can also just install Ollama from https://ollama.com/ beforehand; the setup cell will detect it.)

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_6-18.jpg?raw=true)

You only need to download Ollama (https://ollama.com/) and follow the instructions in just few minutes your Ollama server will bve ready.

With Ollama installed you just need to install de Python API and load it in the notebook.

Once installed you can Pull the Modeles you are interested in using the command:
**ollama pull < model_name >.**

pere@Peres-MBP ~ % ollama pull llama3

*pulling manifest
pulling ef311de6af9d... 100% ▕████████████████████▏ 5.0 GB                         
pulling 097a36493f71... 100% ▕████████████████████▏ 8.4 KB                         
pulling 109037bec39c... 100% ▕████████████████████▏  136 B                         
pulling 65bb16cf5983... 100% ▕████████████████████▏  109 B                         
pulling 0c2a5137eb3c... 100% ▕████████████████████▏  483 B                         
verifying sha256 digest
writing manifest
removing any unused layers
success*


In [ ]:
%pip install -q ollama

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


In [ ]:
# === ollama bootstrap (bban4040) ===
# Make the notebook self-contained: ensure an ollama server is running and the
# model is pulled. Works on Colab and on a Linux/HPC node.
#
# We install the standalone ollama bundle directly instead of the official
# `curl https://ollama.com/install.sh | sh` script, which fails in two common
# ways: it needs root+sudo (so it errors on an HPC node), and recent releases
# ship a zstd-compressed bundle that makes the script abort with "requires zstd
# for extraction" on Colab runtimes that lack zstd. The standalone tarball
# extracts anywhere, needs no root or systemd, and still bundles the CUDA libs
# so a GPU is used when present.
import os, sys, time, shutil, socket, platform, subprocess

OLLAMA_MODEL = "llama3.2:3b"
os.environ.setdefault("OLLAMA_HOST", "127.0.0.1:11434")
_scratch = os.environ.get("SCRATCH") or os.path.join("/cluster/work", os.environ.get("USER", ""))
if os.path.isdir(_scratch):
    os.environ.setdefault("OLLAMA_MODELS", os.path.join(_scratch, "ollama_models"))

def _server_up():
    try:
        with socket.create_connection(("127.0.0.1", 11434), timeout=1):
            return True
    except OSError:
        return False

# 1. Locate or install the ollama binary.
ollama_bin = shutil.which("ollama")
if not ollama_bin:
    _dest = os.path.expanduser("~/.local/ollama")
    _cand = os.path.join(_dest, "bin", "ollama")
    if not os.path.exists(_cand):
        # Fresh install. The bundle is zstd-compressed, so make sure a zstd
        # extractor is available first (apt when root on Colab, else the pip
        # 'zstandard' wheel which needs no root).
        _arch = "arm64" if platform.machine() in ("aarch64", "arm64") else "amd64"
        _url = "https://ollama.com/download/ollama-linux-%s.tar.zst" % _arch
        os.makedirs(_dest, exist_ok=True)
        print("Installing ollama (standalone) into %s ..." % _dest)
        if shutil.which("zstd") is None and hasattr(os, "geteuid") and os.geteuid() == 0:
            subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                           shell=True, check=False)
        if shutil.which("zstd"):
            subprocess.run("curl -fsSL %s | zstd -dc | tar -xf - -C %s" % (_url, _dest),
                           shell=True, check=True)
        else:
            # No zstd binary and no root to install it: decompress in Python.
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zstandard"], check=True)
            import urllib.request, tarfile, zstandard
            with urllib.request.urlopen(_url) as _resp, \
                    zstandard.ZstdDecompressor().stream_reader(_resp) as _stream, \
                    tarfile.open(fileobj=_stream, mode="r|") as _tar:
                _tar.extractall(_dest)
    ollama_bin = _cand
    os.environ["PATH"] = os.path.dirname(ollama_bin) + os.pathsep + os.environ["PATH"]

# 2. Start the server in the background if it is not already up.
if not _server_up():
    print("Starting ollama server ...")
    subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(90):
        if _server_up():
            break
        time.sleep(1)
print("ollama server ready:", _server_up())

# 3. Pull the model (no-op if already present).
subprocess.run([ollama_bin, "pull", OLLAMA_MODEL], check=True)
print("model ready:", OLLAMA_MODEL)

In [ ]:
import ollama

Now is necessary to setup a user_message and instruction for the prompt.

In [ ]:
# Define the user message to send.
user_message = "What is the name of the best paid employee?"

In [ ]:
model_instructions = """
Your task is to convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word to appropriately answer the question.
- **Return Only SQL Code.
- **Don't add any explanation.
   ### Input
   This SQL query generatd will run on a database whose schema is represented below:

   create table employees(
       ID_Usr INT primary key,-- Unique Id for employee
       name VARCHAR -- Name of employee
       );

   create table salary(
       ID_Usr INT,-- Unique Id for employee
       year DATE, -- Date
       salary FLOAT, --Salary of employee
       foreign key (ID_Usr) references employees(ID_Usr) -- Join Employees with salary
       );

   create table studies(
       ID_study INT, -- Unique ID study
       ID_Usr INT, -- ID employee
       educational_level INT,  -- 5=phd, 4=Master, 3=Bachelor
       Institution VARCHAR, --Name of instituon where eployee studied
       Years DATE, -- Date acomplishement stdy
       Speciality VARCHAR, -- Speciality of studies
       primary key (ID_study, ID_Usr), --Primary Key ID_Usr + ID_Study
       foreign key(ID_Usr) references employees (ID_Usr)
       );
"""

In [ ]:
# Call a base model directly with system instructions + the user question.
# (The original notebook used a custom 'myllamasql' model that was never
# created. We use 'llama3.2:3b' — a small base model pulled with
# `ollama pull llama3.2:3b`. Swap in 'llama3' if you have >=6 GB RAM free.)
# num_predict caps output length (SQL answers are short) to keep CPU latency low.
response = ollama.generate(model='llama3.2:3b',
                           system=model_instructions,
                           prompt=user_message,
                           options={'num_predict': 256})
print(response['response'])

As you noticed you are passing the user question and the model instructions in two different variables. This is due to the template that ollama use to call the model. 

{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>



You can create a new model based in one existent and override the variables, to change how the model works. 

In [ ]:
#Configuration based on llama3.2:3b and using a One Shot Sample.
#(For reference only — the create() call below uses the structured ollama API.)
modelfile = """
FROM llama3.2:3b

MESSAGE user How Many employees we have with a salary bigger than 50000?
MESSAGE assistant SELECT COUNT(*) AS total_employees FROM employees e INNER JOIN salary s ON e.ID_Usr = s.ID_Usr WHERE s.salary > 50000;

PARAMETER repeat_penalty 1.2
PARAMETER temperature 0.1
"""

In [ ]:
# Creating a new model with different hyperparameters and a one-shot sample.
# Newer ollama-python (>=0.4) replaced the `modelfile=` string argument with
# structured keywords: from_ (base model), messages (the shots) and parameters.
ollama.create(
    model="llamasql",
    from_="llama3.2:3b",
    messages=[
        {"role": "user",
         "content": "How Many employees we have with a salary bigger than 50000?"},
        {"role": "assistant",
         "content": "SELECT COUNT(*) AS total_employees FROM employees e "
                    "INNER JOIN salary s ON e.ID_Usr = s.ID_Usr WHERE s.salary > 50000;"},
    ],
    parameters={"repeat_penalty": 1.2, "temperature": 0.1},
)

In [ ]:
response = ollama.generate(model='llamasql',
                           system=model_instructions,
                           prompt=user_message,
                           options={'num_predict': 256})

In [ ]:
print(response['response'])

Is posible to create a new model with a file  stored in a directory. 

Sample of file: 
_____________
FROM llama3

SYSTEM """ Your task is to convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word to appropriately answer the question.
- **Return Only SQL Code. 
- **Don't add any explanation.
   
   This SQL query generatd will run on a database whose schema is represented below:

   create table employees(
       ID_Usr INT primary key,-- Unique Id for employee
       name VARCHAR -- Name of employee
       );

   create table salary(
       ID_Usr INT,-- Unique Id for employee
       year DATE, -- Date
       salary FLOAT, --Salary of employee
       foreign key (ID_Usr) references employees(ID_Usr) -- Join Employees with salary
       );

   create table studies(
       ID_study INT, -- Unique ID study
       ID_Usr INT, -- ID employee
       educational_level INT,  -- 5=phd, 4=Master, 3=Bachelor
       Institution VARCHAR, --Name of instituon where eployee studied
       Years DATE, -- Date acomplishement stdy
       Speciality VARCHAR, -- Speciality of studies
       primary key (ID_study, ID_Usr), --Primary Key ID_Usr + ID_Study
       foreign key(ID_Usr) references employees (ID_Usr)
       );"""

MESSAGE user How Many employees we have with a salary bigger than 50000?

MESSAGE assistant """
SELECT COUNT(*) AS total_employees
FROM employees e
INNER JOIN salary s ON e.ID_Usr = s.ID_Usr
WHERE s.salary > 50000;"""

MESSAGE user Return the names of the three people who have had the highest salary increase in the last three years.

MESSAGE assistant """
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year >= DATE_SUB(CURDATE(), INTERVAL 3 YEAR)
GROUP BY e.name
ORDER BY (MAX(s.salary) - MIN(s.salary)) DESC
LIMIT 3;"""

PARAMETER repeat_penalty 1.2
PARAMETER temperature 0.1
_____________________________

This file contains the system instructions, a pair of shots and two hyperparameters. To create the Model you can use the order: 

**ollama create llamanlsql -f < filename >**
from your terminal. 


In [ ]:
# The markdown above shows how to build a model from a Modelfile via the
# terminal (`ollama create llamanlsql -f <file>`). Here we build the equivalent
# model in code so the notebook is self-contained, baking the system message and
# two shots into the model itself.
ollama.create(
    model="llamanlsql",
    from_="llama3.2:3b",
    system=model_instructions,
    messages=[
        {"role": "user",
         "content": "How Many employees we have with a salary bigger than 50000?"},
        {"role": "assistant",
         "content": "SELECT COUNT(*) AS total_employees\nFROM employees e\n"
                    "INNER JOIN salary s ON e.ID_Usr = s.ID_Usr\nWHERE s.salary > 50000;"},
        {"role": "user",
         "content": "Return the names of the three people who have had the "
                    "highest salary increase in the last three years."},
        {"role": "assistant",
         "content": "SELECT e.name\nFROM employees e\n"
                    "JOIN salary s ON e.ID_usr = s.ID_usr\n"
                    "WHERE s.year >= DATE_SUB(CURDATE(), INTERVAL 3 YEAR)\n"
                    "GROUP BY e.name\nORDER BY (MAX(s.salary) - MIN(s.salary)) DESC\nLIMIT 3;"},
    ],
    parameters={"repeat_penalty": 1.2, "temperature": 0.1},
)

# Notice you don't need to pass the system message now: it is baked into the model.
response = ollama.generate(model='llamanlsql',
                           prompt=user_message,
                           options={'num_predict': 256})

In [ ]:
print(response['response'])